In [5]:
train_data_path = "../data/train_data.jsonl"
dev_data_path = "../data/dev_data.jsonl"
test_data_path = "../data/test_data.jsonl"

In [7]:
import pandas as pd

train_data = pd.read_json(train_data_path, lines=True)[['text', 'label']]
dev_data = pd.read_json(dev_data_path, lines=True)[['text', 'label']]
test_data = pd.read_json(test_data_path, lines=True)[['text', 'label']]

In [5]:
import os
import json
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib
from sklearn.pipeline import FeatureUnion


# Combine train and dev sets for training
train_dev_data = pd.concat([train_data, dev_data], ignore_index=True)


vectorizer = FeatureUnion([
    ("word_tfidf", TfidfVectorizer(
        lowercase=False,
        max_features=20000,
        ngram_range=(1, 2)
    )),
    ("char_tfidf", TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        max_features=10000,
        lowercase=False
    ))
])
X_train_dev = vectorizer.fit_transform(train_dev_data['text'])
X_test = vectorizer.transform(test_data['text'])

Y_train_dev = train_dev_data['label']
Y_test = test_data['label']

rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_classifier.fit(X_train_dev, Y_train_dev)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [6]:
test_predictions = rf_classifier.predict(X_test)

test_accuracy = accuracy_score(Y_test, test_predictions)
test_precision = precision_score(Y_test, test_predictions, average='binary')
test_recall = recall_score(Y_test, test_predictions, average='binary')
test_f1 = f1_score(Y_test, test_predictions, average='binary')

print("Test Set Metrics:")
print(f"Accuracy: {test_accuracy:.2f}")
print(f"Precision: {test_precision:.2f}")
print(f"Recall: {test_recall:.2f}")
print(f"F1 Score: {test_f1:.2f}")
print("\nTest Classification Report:\n")
print(classification_report(Y_test, test_predictions))

Test Set Metrics:
Accuracy: 0.89
Precision: 0.89
Recall: 0.89
F1 Score: 0.89

Test Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.88      0.88     16272
           1       0.89      0.89      0.89     18000

    accuracy                           0.89     34272
   macro avg       0.89      0.89      0.89     34272
weighted avg       0.89      0.89      0.89     34272



In [ ]:
save_dir = "../saved_model"
os.makedirs(save_dir, exist_ok=True)

joblib.dump(vectorizer, os.path.join(save_dir, "vectorizer.pkl"))
joblib.dump(rf_classifier, os.path.join(save_dir, "rf_model.pkl"))


In [8]:
import joblib
import pandas as pd

load_dir = "../saved_model"

vectorizer = joblib.load(os.path.join(load_dir, "vectorizer.pkl"))
rf_classifier = joblib.load(os.path.join(load_dir, "rf_model.pkl"))

print("Model loaded successfully.")

/Users/m1/Desktop/HumanVsAI_Text_Classification/myenv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/m1/Desktop/HumanVsAI_Text_Classification/myenv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/m1/Desktop/HumanVsAI_Text_Classification/myenv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWa

Model loaded successfully.


/Users/m1/Desktop/HumanVsAI_Text_Classification/myenv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [10]:
def predict_text(texts):
    """
    texts: list of strings or single string
    """
    if isinstance(texts, str):
        texts = [texts]

    X = vectorizer.transform(texts)
    preds = rf_classifier.predict(X)
    return preds


# Example
samples = [
"Distance education has become increasingly popular in recent years as an alternative method of instruction that allows individuals who may not be able to attend regular classes due to work schedules, family obligations, physical limitations (e.g., mobility issues), and\/or financial constraints.  Distance education also offers several advantages when compared to traditional face-to-face classrooms such as:  Students are no longer required to physically travel from one location to another which eliminates time spent traveling between home\/work\/school; this is especially beneficial if they live too far away from campus or have limited transportation options available.   In addition, many online courses offer flexible scheduling so students do not need to commit themselves exclusively to attending certain times throughout each week since there will always be some flexibility built into the course schedule based upon individual student needs.  \nStudents enrolled in online programs often report higher levels of satisfaction than those participating in conventional educational environments because they feel less stressed about being late to class, missing important information presented during lectures\/tutorials,..."
]

print(predict_text(samples))

[1]


<>:15: SyntaxWarning: invalid escape sequence '\/'
<>:15: SyntaxWarning: invalid escape sequence '\/'
/var/folders/73/lwz93zqs6csf77m39pg7cxz00000gn/T/ipykernel_80768/95549913.py:15: SyntaxWarning: invalid escape sequence '\/'
  "Distance education has become increasingly popular in recent years as an alternative method of instruction that allows individuals who may not be able to attend regular classes due to work schedules, family obligations, physical limitations (e.g., mobility issues), and\/or financial constraints.  Distance education also offers several advantages when compared to traditional face-to-face classrooms such as:  Students are no longer required to physically travel from one location to another which eliminates time spent traveling between home\/work\/school; this is especially beneficial if they live too far away from campus or have limited transportation options available.   In addition, many online courses offer flexible scheduling so students do not need to commi